In [0]:
%sql
SELECT
    priority,
    referral_status,
    appointment_status,
    COUNT(*) AS patient_count,
    ROUND(AVG(waiting_days), 1) AS avg_waiting_days,
    MAX(waiting_days) AS max_waiting_days
FROM healthcare_demo.silver.patient_referral_details
GROUP BY priority, referral_status, appointment_status
ORDER BY priority;

In [0]:
%sql
SELECT DISTINCT
    location_id,
    location_name,
    specialty,
    available_slots
FROM healthcare_demo.silver.patient_referral_details
ORDER BY location_id, specialty;

In [0]:
%sql
SELECT
    patient_id,
    referral_id,
    priority,
    specialty,
    location_id,
    location_name,
    referral_date,
    waiting_days,
    appointment_status
FROM healthcare_demo.silver.patient_referral_details
WHERE appointment_id IS NULL
   OR LOWER(appointment_status) IN ('cancelled', 'waiting')
ORDER BY waiting_days DESC;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_table = "healthcare_demo.silver.patient_referral_details"

df = spark.table(silver_table)

# Extract confirmed hospital capacity
capacity_df = (
    df.select(
        "location_id",
        "specialty",
        "available_slots"
    )
    .filter(
        F.col("available_slots").isNotNull() &
        (F.col("available_slots") > 0)
    )
    .dropDuplicates(["location_id", "specialty"])
)

# Add hospital names using location IDs
capacity_df = capacity_df.withColumn(
    "location_name",
    F.when(F.col("location_id") == "L001", "Central Clinic")
    .when(F.col("location_id") == "L002", "North Clinic")
    .when(F.col("location_id") == "L003", "South Clinic")
    .otherwise("Unknown Clinic")
)

display(capacity_df)

In [0]:
waiting_df = (
    df.filter(
        (F.col("appointment_id").isNull()) |
        (F.lower(F.col("appointment_status")).isin(
            "cancelled", "waiting"
        ))
    )
    .select(
        "patient_id",
        "referral_id",
        "priority",
        "specialty",
        "referral_date",
        "waiting_days",
        "clinical_symptoms",
        "document_priority",
        "document_specialty"
    )
    .dropDuplicates(["referral_id"])
)

# Rank referrals by priority and waiting time
priority_window = Window.partitionBy("specialty").orderBy(
    F.when(F.lower(F.col("priority")) == "urgent", 1).otherwise(2),
    F.col("waiting_days").desc(),
    F.col("referral_date").asc()
)

waiting_df = waiting_df.withColumn(
    "priority_rank",
    F.row_number().over(priority_window)
)

print(waiting_df.columns)
display(waiting_df)

In [0]:
# Prepare capacity columns for joining
# Prepare capacity columns for joining
capacity_for_join = (
    capacity_df
    .withColumn(
        "recommended_location_name",
        F.when(F.col("location_id") == "L001", "Central Clinic")
        .when(F.col("location_id") == "L002", "North Clinic")
        .when(F.col("location_id") == "L003", "South Clinic")
        .otherwise("Unknown Clinic")
    )
    .select(
        F.col("specialty").alias("capacity_specialty"),
        F.col("location_id").alias("recommended_location_id"),
        "recommended_location_name",
        "available_slots"
    )
)

display(capacity_for_join)

# Match patients to hospitals offering their specialty
allocation_df = (
    waiting_df.alias("w")
    .join(
        capacity_for_join.alias("c"),
        F.col("w.specialty") == F.col("c.capacity_specialty"),
        "left"
    )
    .select(
        "w.*",
        F.col("c.recommended_location_id"),
        F.col("c.recommended_location_name"),
        F.col("c.available_slots")
    )
)

# Determine recommendation status
allocation_df = allocation_df.withColumn(
    "allocation_status",
    F.when(
        F.col("available_slots").isNull(),
        "Capacity information unavailable"
    ).when(
        F.col("priority_rank") <= F.col("available_slots"),
        "Recommended"
    ).otherwise(
        "Waiting for capacity"
    )
)

# Add recommendation explanation
allocation_df = allocation_df.withColumn(
    "recommendation_reason",
    F.when(
        F.col("allocation_status") == "Recommended",
        F.concat(
            F.lit("Specialty match and confirmed capacity. Priority rank: "),
            F.col("priority_rank").cast("string")
        )
    ).otherwise(F.col("allocation_status"))
)

allocation_df = allocation_df.withColumn(
    "recommended_at",
    F.current_timestamp()
)

display(allocation_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_demo.gold")

(
    allocation_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_demo.gold.appointment_recommendations")
)

print("Patient allocation recommendations saved successfully")

In [0]:
%sql
SELECT
    specialty,
    recommended_location_name,
    allocation_status,
    COUNT(*) AS patient_count
FROM healthcare_demo.gold.appointment_recommendations
GROUP BY
    specialty,
    recommended_location_name,
    allocation_status
ORDER BY specialty;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Rank patients within each specialty
priority_window = (
    Window
    .partitionBy("specialty")
    .orderBy(
        F.when(F.lower(F.col("priority")) == "urgent", 1).otherwise(2),
        F.col("waiting_days").desc(),
        F.col("referral_date").asc()
    )
)

waiting_ranked = waiting_df.withColumn(
    "priority_rank",
    F.row_number().over(priority_window)
)

# Prepare capacity records with clinic names
capacity_for_join = (
    capacity_df
    .withColumn(
        "location_name",
        F.when(F.col("location_id") == "L001", "Central Clinic")
        .when(F.col("location_id") == "L002", "North Clinic")
        .when(F.col("location_id") == "L003", "South Clinic")
        .otherwise("Unknown Clinic")
    )
    .select(
        "specialty",
        F.col("location_id").alias("recommended_location_id"),
        F.col("location_name").alias("recommended_location_name"),
        "available_slots"
    )
)

# Match patients to their specialty clinic
allocation_df = waiting_ranked.join(
    capacity_for_join,
    on="specialty",
    how="left"
)

# Allocate recommendations based on available capacity
allocation_df = (
    allocation_df
    .withColumn(
        "allocation_status",
        F.when(
            F.col("available_slots").isNull(),
            "Capacity information unavailable"
        ).when(
            F.col("priority_rank") <= F.col("available_slots"),
            "Recommended"
        ).otherwise("Waiting for capacity")
    )
    .withColumn(
        "recommendation_reason",
        F.concat(
            F.lit("Specialty match; priority: "),
            F.coalesce(F.col("priority"), F.lit("Unknown")),
            F.lit("; waiting days: "),
            F.coalesce(
                F.col("waiting_days").cast("string"),
                F.lit("Unknown")
            )
        )
    )
    .withColumn(
        "recommended_at",
        F.current_timestamp()
    )
)

# Display allocation recommendations
display(allocation_df)

In [0]:
display(
    allocation_df
    .groupBy("specialty", "allocation_status")
    .count()
    .orderBy("specialty", "allocation_status")
)

In [0]:

display(
    allocation_df
    .filter(F.col("allocation_status") == "Recommended")
    .select(
        "patient_id",
        "referral_id",
        "priority",
        "specialty",
        "waiting_days",
        "recommended_location_id",
        "recommended_location_name",
        "allocation_status",
        "recommendation_reason"
    )
    .orderBy(
        "specialty",
        "priority_rank"
    )
)

In [0]:
allocation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.appointment_recommendations")

print("Patient recommendations saved successfully")

In [0]:
saved_recommendations = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

display(
    saved_recommendations
    .groupBy("specialty", "allocation_status")
    .count()
    .orderBy("specialty", "allocation_status")
)

In [0]:

from pyspark.sql import functions as F

healthcare_report = (
    allocation_df
    .groupBy("specialty")
    .agg(
        F.first("recommended_location_id", ignorenulls=True).alias("location_id"),
        F.first("recommended_location_name", ignorenulls=True).alias("location_name"),
        F.max("available_slots").alias("available_slots"),
        F.countDistinct("referral_id").alias("total_waiting_referrals"),
        F.round(F.avg("waiting_days"), 1).alias("average_waiting_days"),
        F.max("waiting_days").alias("maximum_waiting_days"),
        F.sum(
            F.when(F.col("allocation_status") == "Recommended", 1)
             .otherwise(0)
        ).alias("recommended_patients"),
        F.sum(
            F.when(F.col("allocation_status") == "Waiting for capacity", 1)
             .otherwise(0)
        ).alias("patients_waiting_for_capacity")
    )
    .orderBy("specialty")
)

display(healthcare_report)


In [0]:
healthcare_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_demo.gold.healthcare_allocation_summary")

print("Final healthcare allocation report saved successfully")

In [0]:
# Clinical information in patient allocation recommendations

clinical_allocation_report = (
    allocation_df
    .groupBy(
        "clinical_symptoms",
        "document_priority",
        "document_specialty",
        "allocation_status"
    )
    .agg(
        F.countDistinct("referral_id").alias("total_referrals"),
        F.round(
            F.avg("waiting_days"), 1
        ).alias("average_waiting_days")
    )
    .orderBy(
        "document_specialty",
        "document_priority",
        "allocation_status"
    )
)

display(clinical_allocation_report)

# Save Gold table

clinical_allocation_report.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_demo.gold.clinical_allocation_summary"
    )

print("Clinical allocation summary saved successfully")

In [0]:

from pyspark.sql import functions as F

# Read Gold recommendations
recommendations_df = spark.table(
    "healthcare_demo.gold.appointment_recommendations"
)

# Check duplicate referral recommendations
duplicate_allocations = (
    recommendations_df
    .groupBy("referral_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check recommendations against clinic capacity
capacity_validation = (
    recommendations_df
    .filter(F.col("allocation_status") == "Recommended")
    .groupBy(
        "specialty",
        "recommended_location_id"
    )
    .agg(
        F.countDistinct("referral_id").alias("recommended_patients"),
        F.max("available_slots").alias("available_slots")
    )
    .withColumn(
        "result",
        F.when(
            F.col("recommended_patients") <= F.col("available_slots"),
            "PASS"
        ).otherwise("FAIL")
    )
)

# Check missing clinic recommendations
missing_clinic = recommendations_df.filter(
    (F.col("allocation_status") == "Recommended") &
    (
        F.col("recommended_location_id").isNull() |
        F.col("recommended_location_name").isNull()
    )
).count()

# Display capacity validation
display(capacity_validation)

# Display overall allocation checks
allocation_results = [
    ("Duplicate referral allocations", duplicate_allocations),
    ("Recommended patients missing clinic", missing_clinic),
    (
        "Capacity violations",
        capacity_validation.filter(F.col("result") == "FAIL").count()
    )
]

allocation_quality_df = spark.createDataFrame(
    allocation_results,
    ["quality_check", "failed_records"]
).withColumn(
    "result",
    F.when(F.col("failed_records") == 0, "PASS")
     .otherwise("FAIL")
)

display(allocation_quality_df)

if allocation_quality_df.filter(F.col("result") == "FAIL").count() == 0:
    print("Gold allocation validation PASSED")
else:
    print("Gold allocation validation FAILED")
